UPscaledEV Project Main Code
Under the funding from TotalEnergies
Author: Yizhan Gu
Email: yig031@ucsd.edu
Affiliation: UCSD CER

Readme:
This code aims at formulating a complex optimization framework with DERs of building, EV, PV and BESS, to solve a value-stacking problem with wholesale market and demand response market participation. It's based on Yi-An Chen's sclaedev codd and more consice, robust and well-constructed.

Labels:
NOTE: means there's a note and please read it
FIXME: means it's a bug or a problem that needs solving
TODO: means it's a to-do task, but not critical to the code running
VERSION: means there're more than one version for the diversity purpose, possibly shows in objective function choices or results analyses

Acknowledgement:
I gratefully acknowledge the support from my PI Jan Kleissl and TotalEnergies team, and the contributions from Yi-An Chen, whose prior work laid the foundation for this project. Special thanks to the UCSD Grid Lab team members.

Set working path and import packages

In [1]:

import os
import os.path
os.chdir('/Users/admin/Desktop/EV_program/Total Transfer/PowerFlex_Code/UPSCALeDEV_2024') 
print("Path is:", os.getcwd(), "\n")

import pandas as pd
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
 
from datetime import timedelta
from datetime import datetime
import time
from time import process_time
import calendar
import holidays
from pathlib import Path
import cvxpy as cp
import sys
from forecast_ED_PD import KnownUser, UnKnownUser

Path is: /Users/admin/Desktop/EV_program/Total Transfer/PowerFlex_Code/UPSCALeDEV_2024 



Parameters

In [3]:

# Plotting colors
Blue1 = (0.341, 0.82, 1)
Blue2 = (0.341, 0.592, 1)
Blue3 = (0.365, 0.365, 1)
Blue4 = (0, 0, 0.89)
Blue5 = (0, 0, 0.639)
Blue6 = (0, 0.161, 0.42)
Red1 = (1, 0.725, 0.635)
Red2 = (1, 0.6, 0.467)
Red3 = (1, 0.361, 0.353)
Red4 = (1, 0.153, 0.137)
Red5 = (0.812, 0.016, 0)
Red6 = (0.541, 0.008, 0)

# MPC parameters
dt_m_EV = 15
dt_h = dt_m_EV/60
interval = timedelta(minutes=dt_m_EV)
unit = dt_h

# Energy Prices-AL-TOU:
# Summer: off-peak period energy charge rate $0.10679/kW h
#         On-peak period energy charge rate $0.12628/kW h
# Winter: off-peak period energy charge rate $0.09506/kW h
#         On-peak period energy charge rate $0.10626/kW h
c_e_TOU_AL = np.ones((96, 1))*0.10679
c_e_TOU_AL[16*4:21*4] = 0.12628
# TODO: Add Winter rates and auto for days


# Demand Charges
c_NCD = 24.48  #24.48*0.25=6.12
c_PD = 19.14 + 9.78

# Tax
c_tax_DWR = 0.00580                               # x Total kWh
c_tax_oEESbo_Franchise = 0.0688 * c_tax_DWR       # x Total kWh
c_tax_CA_Surcharge = 0.00030                      # x Total kWh
c_tax_CA_Regulatory = 0.00058                     # x Total kWh
# x Total Bill (UDC+Commodity)
c_tax_SD_Franchise = 0.0578
c_tax_all = c_tax_DWR + c_tax_oEESbo_Franchise + c_tax_CA_Surcharge + c_tax_CA_Regulatory

Forecast methods

In [ ]:

Numb_EVs = 0
Numb_AbsDiffEVs = 0

Fc_SessionkWh = 'PerfectSessionkWh'  # 'PerfectSessionkWh' #'PersistenceSessionkWh'
Fc_NumbEV = 'PerfectNumbEV'          # 'PerfectNumbEV'     #'PersistenceNumbEV'
Fc_AtArrival = 'PerfectatArrival'    # 'PerfectatArrival'  #'MLatArrival' from Avik
Fc_building = 'Perfect'              # 'Perfectat'  #'Persistence'
Fc_PV = 'Perfect'                    # 'Perfectat'  #'Persistence'

DAM = 1 # 1/0: w/o day-ahead market participation (Demand Response) # FIXME: delete

#always run both the base case (no demand reduction) and the case1 with service reduction # FIXME: rename
Cases = ['Base', 'Case1']

Data processing

In [7]:

# TODO: use 2025 data
Data = pd.read_csv('/Users/admin/Desktop/EV_program/Total Transfer/PowerFlex_Data/UCSD_AllSites_Merge_Eta_20210504_20230129_v3.csv', low_memory=False)

Data['Interval start'] = pd.to_datetime(Data['Interval start'])
Data['Interval end'] = pd.to_datetime(Data['Interval end'])
Data['Session start'] = pd.to_datetime(Data['Session start'])
Data['Session end'] = pd.to_datetime(Data['Session end'])

if Fc_AtArrival == 'MLatArrival':
    User_Data = pd.read_csv("/Users/admin/Desktop/EV_program/Total Transfer/Forecasting/Driver Table and Sessions/Driver_Table.csv")
    User_known = User_Data['driver_id'][User_Data['TotSession']>10].unique()
    UserNoBess = []

year = 2022

# create weekdays (excluding holidays) and weekends (including holidays) lists
Holidays = holidays.US()
Holidays_dates = list(Holidays.keys())
Holidays_dates = [date for date in Holidays_dates if date.year == year]

datetime(year,1,1) in Holidays
start_ind_Y = datetime(year,1,1)
end_ind_Y = datetime(year+1,1,1)
DateSeries_ThisY = []
while start_ind_Y < end_ind_Y:
    DateSeries_ThisY.append(start_ind_Y)
    start_ind_Y += timedelta(hours=24)

DateSeries_ThisY = pd.Series(DateSeries_ThisY)
Y_Weekends = DateSeries_ThisY[DateSeries_ThisY.dt.dayofweek>=5]
Y_Holidays = DateSeries_ThisY[DateSeries_ThisY.dt.date.isin(Holidays_dates)]

Y_WeekendsWH = pd.concat([Y_Weekends,Y_Holidays],axis=0).drop_duplicates(keep='first', inplace=False).sort_values(axis=0, ascending=True).reset_index(drop=True)
Y_WeekdaysWOH = DateSeries_ThisY[~(DateSeries_ThisY.isin(Y_WeekendsWH))].sort_values(axis=0, ascending=True).reset_index(drop=True)
if len(Y_WeekdaysWOH) + len(Y_WeekendsWH) != len(DateSeries_ThisY):
    print("Error in holiday identification!\n")
    sys.exit()

# TODO: add more filter of 2025 data
Data = Data[Data['Interval start'].dt.year == 2022]
print("Number of Intervals:", len(Data))
print("Date range:", Data['Interval start'].min(), "to", Data['Interval start'].max())
print("Unique sites:", Data['Site'].unique())








Number of Intervals: 329805
Date range: 2022-01-01 06:45:00 to 2022-12-31 19:30:00
Unique sites: ['Athena Garage' 'UCSD Hopkins JT' 'UCSD Gilman Parking Structure']


Main loop

In [ ]:
for month in np.array(range(1))+12:
    num_days = calendar.monthrange(year,month)[1] 
    
    M_Th_NCD = [0 for _ in range(len(Cases)+1)]
    M_Th_PD = [0 for _ in range(len(Cases)+1)]
    M_Th_NCD_list = [[] for _ in range(len(Cases)+1)]
    M_Th_PD_list = [[] for _ in range(len(Cases)+1)]
    M_Th_NCD_list_96 = [[] for _ in range(len(Cases)+1)]
    M_Th_PD_list_96 = [[] for _ in range(len(Cases)+1)]
    
    
    M_V0G = pd.DataFrame()
    M_V1G = pd.DataFrame()
    M_Opt_Offline = pd.DataFrame()
    M_Opt_Imp = [pd.DataFrame() for _ in range(len(Cases))] #pd.DataFrame()
    M_t = pd.DataFrame()
    
    M_ED_V0G = []
    M_ED_V1G = []
    M_ED_Opt_DA = []
    M_ED_Opt_t0 = []
    M_ED_Opt_base = []
    M_ED_Opt_case1 = []
    
    M_baseline_Opt = pd.DataFrame()
    
    for Day in np.array(range(1))+1:
        TheDate_Day0 = datetime(year, month, Day)   
        TheDate_Day0_Start = TheDate_Day0 + timedelta(seconds=0)
        TheDate_Day0_End = TheDate_Day0 + timedelta(hours=24)
        
        if len(Y_WeekdaysWOH[Y_WeekdaysWOH==TheDate_Day0])>0: 
            Order = Y_WeekdaysWOH[Y_WeekdaysWOH==TheDate_Day0].index  
            TheDate_Dayb1 = Y_WeekdaysWOH[Order-1].iloc[0]
            TheDate_Dayb2 = Y_WeekdaysWOH[Order-2].iloc[0]
            TheDate_Dayb3 = Y_WeekdaysWOH[Order-3].iloc[0]
        else:
            Order = Y_WeekendsWH[Y_WeekendsWH==TheDate_Day0].index  
            TheDate_Dayb1 = Y_WeekendsWH[Order-1].iloc[0]
            TheDate_Dayb2 = Y_WeekendsWH[Order-2].iloc[0]
            TheDate_Dayb3 = Y_WeekendsWH[Order-3].iloc[0]
            

        
        TheDate_Dayb1_Start = TheDate_Dayb1 + timedelta(seconds=0)
        TheDate_Dayb1_End   = TheDate_Dayb1 + timedelta(hours=24)
        TheDate_Dayb2_Start = TheDate_Dayb2 + timedelta(seconds=0)
        TheDate_Dayb2_End   = TheDate_Dayb2 + timedelta(hours=24)
        TheDate_Dayb3_Start = TheDate_Dayb3 + timedelta(seconds=0)
        TheDate_Dayb3_End   = TheDate_Dayb3 + timedelta(hours=24)
        
        TheDate        = [TheDate_Day0, TheDate_Dayb1, TheDate_Dayb2, TheDate_Dayb3]
        TheDates_Start = [TheDate_Day0_Start, TheDate_Dayb1_Start, TheDate_Dayb2_Start, TheDate_Dayb3_Start]   
        TheDates_End   = [TheDate_Day0_End, TheDate_Dayb1_End, TheDate_Dayb2_End, TheDate_Dayb3_End]
        
        
        # NOTE: Procedures in timeline
        Read_LMP()
        Baseline()
        EV_data_choosing()
        EV_forecast()
        DA_optimization()
        RT_optimization()
        Daily_results()
        Daily_cost_analysis()
        
    # End of Day loop
# End of Month loop
        
        

Functions

In [ ]:
def Read_LMP():
    dir_Input_DA = os.path.join("/Users/admin/Desktop/EV_program/18_Jan_2024/Jan_2024_Data/LMP/LMP_DA")
    dir_Input_RT = os.path.join( "/Users/admin/Desktop/EV_program/18_Jan_2024/Jan_2024_Data/LMP/LMP_IN")
    #dir_Input = os.path.join("/Users/admin/Desktop/EV_program/Total Transfer/PowerFlex_Code/UPSCALeDEV_2024/LMP")
    filename_Input = dir_Input_DA + '/LMP_DAM_' + TheDate_Day0_Start.strftime('%Y%m%d') + '.csv'
    Data_LMP_DA = pd.read_csv(filename_Input)
    filename_Input = dir_Input_RT + '/INTVL_LMP_' + TheDate_Day0_Start.strftime('%Y%m%d') + '.csv'
    Data_LMP_RT = pd.read_csv(filename_Input)
    
    # NOTE: The LMP data 2022 is incomplete, please manually select usable days
            
    Data_LMP_DA = Data_LMP_DA[Data_LMP_DA['LMP_TYPE']=='LMP']
    Data_LMP_RT = Data_LMP_RT[Data_LMP_RT['LMP_TYPE']=='LMP']
    
    # sort the 'INTERVALSTARTTIME_GMT'
    LMP_IntervalStartGMT_DA = Data_LMP_DA['INTERVALSTARTTIME_GMT']
    LMP_IntervalStartGMT_RT = Data_LMP_RT['INTERVALSTARTTIME_GMT']
    
    # sort LMP data based on the 'INTERVALSTARTTIME_GMT'    
    A = pd.to_datetime(LMP_IntervalStartGMT_DA)
    B = [k for k in range(len(A))]
    LMPOrder = [x for _, x in sorted(zip(A, B))]
    Data_LMP_DA_ = Data_LMP_DA.iloc[ LMPOrder, :]
    
    A = pd.to_datetime(LMP_IntervalStartGMT_RT)
    B = [k for k in range(len(A))]
    LMPOrder = [x for _, x in sorted(zip(A, B))]
    Data_LMP_RT_ = Data_LMP_RT.iloc[ LMPOrder, :]
    
    
    Bid_Pr_DA = np.array(Data_LMP_DA_['MW'].repeat(4)*0.001).reshape(96, 1)#repeat 4 times of the 24 x 1 series, $/MWh ---> $/kWh   
    Bid_Pr_RT = (np.average(np.array(Data_LMP_RT_['VALUE']).reshape(-1, 3), axis=1)*0.001).reshape(96, 1) #ave every 3 entries of the 288  x 1 series, $/MWh ---> $/kWh
    

In [ ]:
def Baseline():
    start_ind_D = datetime(year, month, Day)
    end_ind_D = start_ind_D + timedelta(hours=24)
    HourSeries_ThisD = []
    while start_ind_D < end_ind_D:
        HourSeries_ThisD.append(start_ind_D)
        start_ind_D += timedelta(hours=1)
            
    # NOTE: The baseline data is actually the history implementation saved in last loop
    # TODO: need to add building and PV and BESS to update the baseline, and first days there are no implementation csvs?
    # 2022 Jan to Nov without DAM and pre saved in another file.
    dir_Input = os.path.join('/Users/admin/Desktop/EV_program/Total Transfer/PowerFlex_Code/2024_RO3.4')
    filename_Input = dir_Input + '/2022_'+Fc_SessionkWh+'_'+Fc_NumbEV+'_'+Fc_AtArrival +'_MonthlyTh_Eta_v5_test4.csv'
    # dir_Input = os.path.join('/Users/admin/Desktop/EV_program/Total Transfer/PowerFlex_Code/UPSCALeDEV_2024/Results/Dispatch')
    # filename_Input = dir_Input + '/2022_'+Fc_SessionkWh+'_'+Fc_NumbEV+'_'+Fc_AtArrival +'_implementation.csv'
    
    Dispatch_2022 = pd.read_csv(filename_Input)
    
    # make sure dropping the earlier data for duplicated rows                
    Dispatch_2022['Interval start'] = pd.to_datetime(Dispatch_2022['Interval start'])
    Dispatch_2022 = Dispatch_2022.drop_duplicates('Interval start',keep='last')


                            
    
    
    # DSGS baseline
    Day0 = TheDate[0]
    Dispatch_day0 = Dispatch_2022[(Dispatch_2022['Interval start'] >= Day0) & (Dispatch_2022['Interval start'] < Day0 + timedelta(days=1))]
    Dispatch_day0 = Dispatch_day0.reset_index()
    
    Eventhour_example = np.zeros(24, dtype=int)
    Eventhour_example[15:20] = 1
    # Only happens 15:00--22:00
    Eventhour_example[0:14] = 0
    Eventhour_example[22:23] = 0

    if Day0 in list(Y_WeekendsWH):
        N_days = 4
    else:
        N_days = 10
    
    Dispatch_preceding = Dispatch_2022[(Dispatch_2022['Interval start'] >= Day0 - timedelta(days=N_days)) & (Dispatch_2022['Interval start'] < Day0)]
    Dispatch_preceding["hms"] = Dispatch_preceding["Interval start"].dt.time

    Load_avg = (
        Dispatch_preceding.groupby("hms")["V1G_real [kWh]"]
        .mean()
        .reset_index()
        .rename(columns={"V1G_real [kWh]": "Load_avg"})
    )
    Load_avg_hourly = Load_avg["Load_avg"].values.reshape(24, 4).mean(axis=1) * 4
    Load_Day0_real_hourly = Dispatch_day0["V1G_real [kWh]"].values.reshape(24, 4).mean(axis=1) * 4
    Load_Day0_V0G_hourly = Dispatch_day0["V0G [kWh]"].values.reshape(24, 4).mean(axis=1) * 4
    Load_Day0_Opt_DA_hourly = Dispatch_day0["Opt_DA [kWh]"].values.reshape(24, 4).mean(axis=1) * 4
    Load_DSGS_hourly = pd.DataFrame({
        "Hour": range(24),
        "EB": Load_avg_hourly,
        "Day0_real": Load_Day0_real_hourly,
        "Event": Eventhour_example,
        "V0G": Load_Day0_V0G_hourly,
        "Opt_DA": Load_Day0_Opt_DA_hourly
    })
    
    start_hour = Load_DSGS_hourly.loc[Load_DSGS_hourly["Event"] == 1, "Hour"].iloc[0]
    DOAV_hours = range(start_hour - 4, start_hour - 1)  # includes hours [start_hour-4, start_hour-2]
    DOAV = sum(Load_DSGS_hourly.loc[DOAV_hours, "Day0_real"]) / sum(Load_DSGS_hourly.loc[DOAV_hours, "EB"])
    if sum(Load_DSGS_hourly.loc[DOAV_hours, "Day0_real"]) < 0 or sum(Load_DSGS_hourly.loc[DOAV_hours, "EB"]) < 0:
        DOAV = 1
    elif DOAV < 0.6:
        DOAV = 0.6
    elif DOAV > 1.4:
        DOAV = 1.4
        
    Load_DSGS_hourly["AEB"] = DOAV * Load_DSGS_hourly["EB"]
    
    y_top = Load_DSGS_hourly[["Day0_real", "V0G", "Opt_DA", "AEB"]].max().max() * 1.05

    # Filter to only event rows
    df_event = Load_DSGS_hourly[Load_DSGS_hourly["Event"] == 1]

    

    
    # Plotting
    plt.figure(figsize=(12, 6))
    plt.plot(Load_DSGS_hourly["Hour"], Load_DSGS_hourly["Day0_real"], label="Day0 (Real Load)", marker='o')
    plt.plot(Load_DSGS_hourly["Hour"], Load_DSGS_hourly["V0G"], label="V0G", marker='s')
    plt.plot(Load_DSGS_hourly["Hour"], Load_DSGS_hourly["Opt_DA"], label="Opt_DA", marker='^')
    plt.plot(Load_DSGS_hourly["Hour"], Load_DSGS_hourly["AEB"], label="AEB (Baseline)", linestyle='--')
    plt.plot(df_event["Hour"], [y_top] * len(df_event), linestyle='', marker='x', color='lightcoral', label="Event Hour", markersize=12, markeredgewidth=2)

    plt.xlabel(f"Hour of {Day0.strftime('%Y-%m-%d')}", fontsize=16)
    plt.ylabel("Load [kW]", fontsize = 16)
    plt.title("Day0 Load & DSGS Baseline", fontsize = 18)
    plt.grid(False)
    plt.xticks(fontsize=14)
    plt.yticks(fontsize=14)
    plt.legend(fontsize=14)
    plt.legend(fontsize=14)
    plt.tight_layout()
    plt.savefig("DSGS_baseline.png", dpi=300)
    plt.show()